# Semantic segmentation with Transformers

Follow the Hugging Face Transformers semantic segmentation tutorial while working through this notebook:

https://huggingface.co/docs/transformers/en/tasks/semantic_segmentation

The code below follows that tutorial closely. The main change is the dataset: instead of SceneParse150, this notebook uses the [SUIM underwater semantic segmentation dataset](https://irvlab.cs.umn.edu/resources/suim-dataset).

References:

- Hugging Face semantic segmentation tutorial: https://huggingface.co/docs/transformers/en/tasks/semantic_segmentation
- SUIM dataset: https://irvlab.cs.umn.edu/resources/suim-dataset
- SUIM paper: https://arxiv.org/abs/2004.01241
- SegFormer checkpoint: https://huggingface.co/nvidia/mit-b0


## Setup

Install the dependencies from `requirements.txt` before running the notebook.


In [ ]:
!pip install -q -r https://raw.githubusercontent.com/stefanluedtke/2026-RoOT/main/requirements.txt

## Load SUIM dataset

This repository contains a compact SUIM subset with about 100 image/mask pairs. If you replace it with the full SUIM download, the code below still reads the same folder structure and limits the number of samples for a feasible first run.

Expected folder layout:

```text
SUIM/
  train_val/
    images/*.jpg
    masks/*.bmp or *.png
  TEST/
    images/*.jpg
    masks/*.bmp or *.png  # direct files; class-specific subfolders are ignored
```


In [ ]:
from pathlib import Path

import numpy as np
from datasets import Dataset, DatasetDict, Image
from PIL import Image as PILImage

suim_root = Path("SUIM")
train_split_name = "train_val"
test_split_name = "TEST"

train_image_dir = suim_root / train_split_name / "images"
train_mask_dir = suim_root / train_split_name / "masks"
test_image_dir = suim_root / test_split_name / "images"
test_mask_dir = suim_root / test_split_name / "masks"


In [ ]:
def list_files(directory, suffixes=(".jpg", ".jpeg", ".png", ".bmp")):
    return sorted(
        path for path in directory.iterdir()
        if path.is_file() and path.suffix.lower() in suffixes
    )


def pair_images_and_masks(image_dir, mask_dir):
    image_paths = list_files(image_dir, suffixes=(".jpg", ".jpeg", ".png"))
    mask_paths = list_files(mask_dir, suffixes=(".png", ".bmp", ".jpg", ".jpeg"))
    masks_by_stem = {path.stem: path for path in mask_paths}

    paired_images = []
    paired_masks = []
    missing = []
    for image_path in image_paths:
        mask_path = masks_by_stem.get(image_path.stem)
        if mask_path is None:
            missing.append(image_path.name)
            continue
        paired_images.append(str(image_path))
        paired_masks.append(str(mask_path))

    if missing:
        raise ValueError(f"Missing masks for {len(missing)} images, for example: {missing[:5]}")
    if not paired_images:
        raise ValueError(f"No image/mask pairs found in {image_dir} and {mask_dir}")
    return paired_images, paired_masks


def create_dataset(image_dir, mask_dir):
    image_paths, mask_paths = pair_images_and_masks(image_dir, mask_dir)
    dataset = Dataset.from_dict({"image": image_paths, "annotation": mask_paths})
    dataset = dataset.cast_column("image", Image())
    dataset = dataset.cast_column("annotation", Image())
    return dataset


ds = DatasetDict({
    "train": create_dataset(train_image_dir, train_mask_dir),
    "test": create_dataset(test_image_dir, test_mask_dir),
})

# Use smaller subsets for a quick first run. Set both values to None for the full dataset.
max_train_samples = 80
max_test_samples = 20

if max_train_samples is not None:
    ds["train"] = ds["train"].shuffle(seed=42).select(range(min(max_train_samples, len(ds["train"]))))
if max_test_samples is not None:
    ds["test"] = ds["test"].shuffle(seed=42).select(range(min(max_test_samples, len(ds["test"]))))

ds


Create the label dictionaries.


In [ ]:
id2label = {
    0: "background_waterbody",
    1: "human_diver",
    2: "plants_seagrass",
    3: "wreck_ruin",
    4: "robot_instrument",
    5: "reef_invertebrate",
    6: "fish_vertebrate",
    7: "seafloor_rock",
}
label2id = {label: str(idx) for idx, label in id2label.items()}
id2label = {str(idx): label for idx, label in id2label.items()}
num_labels = len(id2label)


Convert SUIM color annotations into class-id masks. This replaces the SceneParse150 labels used in the Hugging Face tutorial.


In [ ]:
suim_color_to_id = {
    (0, 0, 0): 0,
    (0, 0, 1): 1,
    (0, 1, 0): 2,
    (0, 1, 1): 3,
    (1, 0, 0): 4,
    (1, 0, 1): 5,
    (1, 1, 0): 6,
    (1, 1, 1): 7,
    (0, 0, 255): 1,
    (0, 255, 0): 2,
    (0, 255, 255): 3,
    (255, 0, 0): 4,
    (255, 0, 255): 5,
    (255, 255, 0): 6,
    (255, 255, 255): 7,
}

suim_palette = np.array([
    [0, 0, 0],
    [0, 0, 255],
    [0, 255, 0],
    [0, 255, 255],
    [255, 0, 0],
    [255, 0, 255],
    [255, 255, 0],
    [255, 255, 255],
], dtype=np.uint8)


def rgb_mask_to_class_ids(mask):
    mask = np.array(mask.convert("RGB"))
    class_mask = np.zeros(mask.shape[:2], dtype=np.uint8)
    for color, class_id in suim_color_to_id.items():
        class_mask[np.all(mask == color, axis=-1)] = class_id
    return PILImage.fromarray(class_mask, mode="L")


def class_ids_to_rgb(mask):
    return suim_palette[np.array(mask, dtype=np.uint8)]


## Task 1

Use the Hugging Face tutorial section **Load SceneParse150 dataset** as a guide.

1. Inspect one example from `ds["train"]`.
2. Display its image.
3. Convert and display its annotation mask.


In [ ]:
# Your code here


## Preprocess

Load the SegFormer image processor and define train and validation transforms.


In [ ]:
from torchvision.transforms import ColorJitter
from transformers import AutoImageProcessor

checkpoint = "nvidia/mit-b0"
image_processor = AutoImageProcessor.from_pretrained(checkpoint, do_reduce_labels=False)

jitter = ColorJitter(brightness=0.25, contrast=0.25, saturation=0.25, hue=0.1)


In [ ]:
def train_transforms(example_batch):
    images = [jitter(image.convert("RGB")) for image in example_batch["image"]]
    labels = [rgb_mask_to_class_ids(mask) for mask in example_batch["annotation"]]
    inputs = image_processor(images=images, segmentation_maps=labels)
    return inputs


def val_transforms(example_batch):
    images = [image.convert("RGB") for image in example_batch["image"]]
    labels = [rgb_mask_to_class_ids(mask) for mask in example_batch["annotation"]]
    inputs = image_processor(images=images, segmentation_maps=labels)
    return inputs


train_ds = ds["train"].with_transform(train_transforms)
test_ds = ds["test"].with_transform(val_transforms)


## Task 2

Use the Hugging Face tutorial section **Preprocess** as a guide.

1. Inspect one transformed training example.
2. Check the shapes of `pixel_values` and `labels`.
3. Explain why `do_reduce_labels=False` is used for SUIM.


In [ ]:
# Your code here


## Evaluate

Load mean IoU and define `compute_metrics`. The logits are resized to the label shape before computing metrics, as in the Hugging Face tutorial.


In [ ]:
import evaluate
import torch
from torch import nn

metric = evaluate.load("mean_iou")


def compute_metrics(eval_pred):
    with torch.no_grad():
        logits, labels = eval_pred
        logits_tensor = torch.from_numpy(logits)
        logits_tensor = nn.functional.interpolate(
            logits_tensor,
            size=labels.shape[-2:],
            mode="bilinear",
            align_corners=False,
        ).argmax(dim=1)

        pred_labels = logits_tensor.detach().cpu().numpy()
        metrics = metric.compute(
            predictions=pred_labels,
            references=labels,
            num_labels=num_labels,
            ignore_index=255,
            reduce_labels=False,
        )

        per_category_accuracy = metrics.pop("per_category_accuracy").tolist()
        per_category_iou = metrics.pop("per_category_iou").tolist()

        metrics.update({f"accuracy_{id2label[str(i)]}": v for i, v in enumerate(per_category_accuracy)})
        metrics.update({f"iou_{id2label[str(i)]}": v for i, v in enumerate(per_category_iou)})
        return metrics


## Train

Load `AutoModelForSemanticSegmentation`, define `TrainingArguments`, and create a `Trainer`.


In [ ]:
from transformers import AutoModelForSemanticSegmentation, TrainingArguments, Trainer

model = AutoModelForSemanticSegmentation.from_pretrained(
    checkpoint,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)


In [ ]:
from inspect import signature

eval_argument_name = "eval_strategy" if "eval_strategy" in signature(TrainingArguments.__init__).parameters else "evaluation_strategy"

training_args = TrainingArguments(
    output_dir="segformer-b0-suim",
    learning_rate=6e-5,
    num_train_epochs=10,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    save_total_limit=3,
    eval_steps=50,
    save_steps=50,
    logging_steps=10,
    eval_accumulation_steps=5,
    remove_unused_columns=False,
    fp16=torch.cuda.is_available(),
    report_to="none",
    push_to_hub=False,
    **{eval_argument_name: "steps"},
)


In [ ]:
trainer_kwargs = dict(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics,
)

if "processing_class" in signature(Trainer.__init__).parameters:
    trainer_kwargs["processing_class"] = image_processor

trainer = Trainer(**trainer_kwargs)


## Task 3

Use the Hugging Face tutorial section **Train** as a guide.

1. Fine-tune the model with `trainer.train()`.
2. Evaluate it with `trainer.evaluate()`.
3. Compare mean IoU after changing the number of training examples or epochs.


In [ ]:
# Your code here


In [ ]:
# Your code here


In [ ]:
# Your code here


## Inference

Run inference on one image and resize the logits to the original image size.


In [ ]:
raw_example = ds["test"][0]
image = raw_example["image"].convert("RGB")
true_mask = rgb_mask_to_class_ids(raw_example["annotation"])

model = AutoModelForSemanticSegmentation.from_pretrained("segformer-b0-suim-final")
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

inputs = image_processor(image, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits.cpu()

upsampled_logits = nn.functional.interpolate(
    logits,
    size=image.size[::-1],
    mode="bilinear",
    align_corners=False,
)
pred_seg = upsampled_logits.argmax(dim=1)[0].numpy().astype(np.uint8)

pred_rgb = class_ids_to_rgb(pred_seg)
true_rgb = class_ids_to_rgb(true_mask)
overlay = (np.array(image) * 0.55 + pred_rgb * 0.45).astype(np.uint8)

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
axes[0].imshow(image)
axes[0].set_title("image")
axes[1].imshow(true_rgb)
axes[1].set_title("ground truth")
axes[2].imshow(pred_rgb)
axes[2].set_title("prediction")
axes[3].imshow(overlay)
axes[3].set_title("prediction overlay")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
